In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import os

import pandas as pd 
import plotly.express as px

# API_TOKEN = "ВАШ_TELEGRAM_BOT_TOKEN"
BOOKS_DIR = "base"
FAISS_DIR = "faiss_index"
INDEX_FILE = os.path.join(FAISS_DIR, "books_index.faiss")
CHUNKS_FILE = os.path.join(FAISS_DIR, "chunks.pkl")


In [ ]:
import ipywidgets as widgets
widgets.IntSlider()


In [ ]:
# chunks = []
# for fname in os.listdir(BOOKS_DIR):
#     if fname.endswith(".txt"):
#         with open(os.path.join(BOOKS_DIR, fname), encoding="utf-8") as f:
#             text = f.read()
#         chunks.extend(chunk_text(text))

In [2]:
TITLES = {
    'SexualPolitics': [
                "FOREWORD", "INTRODUCTION TO THE ILLINOIS PAPERBACK", "INTRODUCTION TO THE TOUCHSTONE PAPERBACK", 
                "PREFACE", "ONE", "TWO", "THREE", "FOUR", "FIVE", 
                "SIX", "SEVEN", "EIGHT", 
                "POSTSCRIPT", "Afterword", "BIBLIOGRAPHY"
                ]
}

In [5]:
import re

# Разбивка на чанки с overlap
def chunk_text(text, chunk_size=1000, overlap=200):
    tokens = text.split()
    chunks = []
    i = 0
    while i < len(tokens):
        chunk = tokens[i:i+chunk_size]
        chunks.append(" ".join(chunk))
        i += chunk_size - overlap
    return chunks

def insert_refs(text, comments):
    pattern = re.compile(r'(?<!\d)([.,;:!?])(\s*)(\d{1,3})(?!\d)')
    ABBREV = {
        'p','pp','vol','vols','no','nos','№','ch','fig','eq',
        'dr','mr','ms','mrs','prof','st','pt','sec','ed','eds','trans','op','cit','ibid','cf'
    }
    def repl(m):
        punct, space, num = m.group(1), m.group(2), m.group(3)

        # контекст слева от знака препинания
        i = m.start()
        left = text[:i]
        # последнее "слово" перед знаком (буквы/№)
        w = re.search(r'([A-Za-zА-Яа-яЁё№]+)\s*$', left)
        token = (w.group(1).lower() if w else '')

        # пропускаем сокращения типа p., No., Vol., ...
        if token in ABBREV:
            return m.group(0)

        note = comments.get(num)
        if note:
            return f"{punct}{space}{num} ({note})"
        else:
            return m.group(0)  # нет примечания — ничего не меняем

    return pattern.sub(repl, text)


# Не универсальная функция
def prepare_text(text, titles):
    text = [i for i in text.split('\n') if i != '']
    text = pd.Series(text)
    print('Абзацев в тексте до обработки: ', text.shape[0])

    title_idx = [int(i) for i in text[text.isin(titles)].loc[67:].index]
    print(title_idx)
    title_start_ends = [(title_idx[i-1], title_idx[i]) for i in range(1, len(title_idx))]

    for start, end in title_start_ends:
        # print(start, end)
        chapter = text.loc[start:end-1]
        
        refs_start = chapter[chapter.str.startswith("1 ")].index
        if refs_start.empty:
            continue
        refs_start = refs_start[0]
        refs_idx = chapter.loc[refs_start:].index
        
        refs = chapter.loc[refs_start:].apply(
                lambda x: (x[:x.find(' ')], x[x.find(' ')+1:])
            ).values
        
        chapter = chapter.loc[:refs_start-1]

        refs = {pair[0]:pair[1] for pair in refs}

        chapter_with_refs = chapter.apply(lambda x: insert_refs(x, refs))
        
        text.loc[chapter_with_refs.index] = chapter_with_refs
        text.drop(index=refs_idx, inplace=True)

    print('Абзацев в тексте после обработки: ', text.shape[0])
    return '\n'.join(text.values)

In [4]:
for fname in os.listdir(BOOKS_DIR):
    if fname.endswith(".txt"):
        with open(os.path.join(BOOKS_DIR, fname), encoding="utf-8") as f:
            text = f.read()
            text = prepare_text(text, titles=TITLES['SexualPolitics'])
            chunks = chunk_text(text)

Абзацев в тексте до обработки:  3791
[67, 105, 119, 132, 142, 277, 464, 1040, 1543, 1973, 2119, 2320, 2520, 2526, 2550]
Абзацев в тексте после обработки:  2641


In [ ]:
len(chunks)

235

In [8]:
# 3. Вычисление эмбеддингов
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks, convert_to_numpy=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\d.abdullina\Documents\pets\llm-dogma-guard\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\d.abdullina\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install ipywidgets jupyterlab_widgets


In [ ]:
!pip install huggingface_hub[hf_xet]


In [ ]:

# 4. Создание FAISS индекса
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# 5. Сохранение индекса и чанков
os.makedirs("faiss_index", exist_ok=True)
faiss.write_index(index, "faiss_index/books_index.faiss")
import pickle
with open("faiss_index/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("FAISS индекс сохранен")
